# SQL Worksheet — Week2

Use the following tables from the Riva Data Platform:

- `rivadataplatform.dataproduct.dim_batch`
- `rivadataplatform.dataproduct.dim_class`
- `rivadataplatform.dataproduct.fact_attendance`
- `rivadataplatform.dataproduct.dim_student`
- `rivadataplatform.dataproduct.dim_date`

**Instructions**
- Write SQL for each question.
- Do not modify the source data.
- Use clear aliases where JOINs are involved.
- Unless a question specifically asks for a particular column, select only the columns needed to answer it.


## Tables / Relationships

Useful keys:
- `dim_student.student_key` ↔ `fact_attendance.student_key`
- `dim_class.class_key` ↔ `fact_attendance.class_key`
- `dim_batch.batch_key` ↔ `fact_attendance.batch_key`
- `dim_class.batch_id` ↔ `dim_batch.batch_id`

## Question 1 — Attendance by Student Location
Join `dim_student`, `fact_attendance`, `dim_class`, and `dim_batch`. Group by city and batch, replacing null values with labels, and return distinct students, total records, and issue records (`Late` or `Absent`). Keep only groups with at least one issue.

In [ ]:
SELECT
    COALESCE(s.city, 'Unknown city') AS city,
    COALESCE(b.batch_name, 'No batch') AS batch_name,
    COUNT(DISTINCT s.student_key) AS distinct_students,
    COUNT(f.attendance_id) AS attendance_records,
    SUM(CASE WHEN f.attendance_status IN ('Late', 'Absent') THEN 1 ELSE 0 END) AS issue_records
FROM rivadataplatform.dataproduct.dim_student AS s
JOIN rivadataplatform.dataproduct.fact_attendance AS f
    ON f.student_key = s.student_key
LEFT JOIN rivadataplatform.dataproduct.dim_class AS c
    ON c.class_key = f.class_key
LEFT JOIN rivadataplatform.dataproduct.dim_batch AS b
    ON b.batch_key = f.batch_key
GROUP BY COALESCE(s.city, 'Unknown city'), COALESCE(b.batch_name, 'No batch')
HAVING SUM(CASE WHEN f.attendance_status IN ('Late', 'Absent') THEN 1 ELSE 0 END) > 0
ORDER BY issue_records DESC, city;

## Question 2 — Class Calendar Status Counts
Join attendance to class, batch, and date dimensions. For each class date and topic, count Present, Late, and Absent records, label null topics, and order by date. Return only classes with attendance records.

In [ ]:
SELECT
    c.class_date,
    COALESCE(d.day_name, c.class_day) AS day_name,
    COALESCE(c.topic, 'Topic not assigned') AS topic,
    b.batch_name,
    SUM(CASE WHEN f.attendance_status = 'Present' THEN 1 ELSE 0 END) AS present_count,
    SUM(CASE WHEN f.attendance_status = 'Late' THEN 1 ELSE 0 END) AS late_count,
    SUM(CASE WHEN f.attendance_status = 'Absent' THEN 1 ELSE 0 END) AS absent_count
FROM rivadataplatform.dataproduct.fact_attendance AS f
JOIN rivadataplatform.dataproduct.dim_class AS c
    ON c.class_key = f.class_key
JOIN rivadataplatform.dataproduct.dim_batch AS b
    ON b.batch_key = f.batch_key
LEFT JOIN rivadataplatform.dataproduct.dim_date AS d
    ON d.date_key = f.date_key
GROUP BY c.class_date, COALESCE(d.day_name, c.class_day), COALESCE(c.topic, 'Topic not assigned'), b.batch_name
HAVING COUNT(f.attendance_id) > 0
ORDER BY c.class_date;

## Question 3 — Null Profile and Attendance Check
Use a `LEFT JOIN` from `dim_student` to attendance and class data. Group by student, replace missing phone and topic values with labels, and return total attendance records plus the number of records whose class topic is missing. Show students with either a missing phone or a missing topic.

In [ ]:
SELECT
    s.student_id,
    s.student_name,
    COALESCE(NULLIF(s.phone_no, ''), 'Phone missing') AS phone_status,
    COUNT(f.attendance_id) AS attendance_records,
    SUM(CASE WHEN c.topic IS NULL OR c.topic = '' THEN 1 ELSE 0 END) AS missing_topic_records
FROM rivadataplatform.dataproduct.dim_student AS s
LEFT JOIN rivadataplatform.dataproduct.fact_attendance AS f
    ON f.student_key = s.student_key
LEFT JOIN rivadataplatform.dataproduct.dim_class AS c
    ON c.class_key = f.class_key
GROUP BY s.student_id, s.student_name, COALESCE(NULLIF(s.phone_no, ''), 'Phone missing')
HAVING NULLIF(s.phone_no, '') IS NULL
    OR SUM(CASE WHEN c.topic IS NULL OR c.topic = '' THEN 1 ELSE 0 END) > 0
ORDER BY s.student_name;

## Question 4 — Batch Attendance Rate
For each batch, calculate total records, distinct students, and the attendance rate where Present or Late counts as attended. Use `NULLIF` to avoid division by zero and return only batches with at least one Absent record.

In [ ]:
SELECT
    b.batch_id,
    b.batch_name,
    COUNT(f.attendance_id) AS attendance_records,
    COUNT(DISTINCT f.student_key) AS distinct_students,
    100.0 * SUM(CASE WHEN f.attendance_status IN ('Present', 'Late') THEN 1 ELSE 0 END)
        / NULLIF(COUNT(f.attendance_id), 0) AS attendance_rate
FROM rivadataplatform.dataproduct.dim_batch AS b
JOIN rivadataplatform.dataproduct.fact_attendance AS f
    ON f.batch_key = b.batch_key
GROUP BY b.batch_id, b.batch_name
HAVING SUM(CASE WHEN f.attendance_status = 'Absent' THEN 1 ELSE 0 END) > 0
ORDER BY attendance_rate DESC;

## Question 5 — Repeated Attendance by Student and Topic
Join students, classes, and attendance, then group by student and null-safe topic. Return students with more than one attendance record for the same topic, including the class date range and the number of Late or Absent records.

In [ ]:
SELECT
    s.student_id,
    s.student_name,
    COALESCE(c.topic, 'Topic not assigned') AS topic,
    COUNT(f.attendance_id) AS attendance_records,
    MIN(c.class_date) AS first_class_date,
    MAX(c.class_date) AS last_class_date,
    SUM(CASE WHEN f.attendance_status IN ('Late', 'Absent') THEN 1 ELSE 0 END) AS issue_records
FROM rivadataplatform.dataproduct.dim_student AS s
JOIN rivadataplatform.dataproduct.fact_attendance AS f
    ON f.student_key = s.student_key
JOIN rivadataplatform.dataproduct.dim_class AS c
    ON c.class_key = f.class_key
GROUP BY s.student_id, s.student_name, COALESCE(c.topic, 'Topic not assigned')
HAVING COUNT(f.attendance_id) > 1
ORDER BY attendance_records DESC, s.student_name;

## Question 6 — Student Present Coverage
Use a `LEFT JOIN` from students to attendance, class, and batch. Group by student and batch to calculate total records, distinct classes, and Present records. Include students with zero attendance and label the batch when it is null.

In [ ]:
SELECT
    s.student_id,
    s.student_name,
    COALESCE(b.batch_name, 'No attendance batch') AS batch_name,
    COUNT(f.attendance_id) AS attendance_records,
    COUNT(DISTINCT f.class_key) AS distinct_classes,
    SUM(CASE WHEN f.attendance_status = 'Present' THEN 1 ELSE 0 END) AS present_count
FROM rivadataplatform.dataproduct.dim_student AS s
LEFT JOIN rivadataplatform.dataproduct.fact_attendance AS f
    ON f.student_key = s.student_key
LEFT JOIN rivadataplatform.dataproduct.dim_batch AS b
    ON b.batch_key = f.batch_key
GROUP BY s.student_id, s.student_name, COALESCE(b.batch_name, 'No attendance batch')
ORDER BY present_count DESC, s.student_name;